In [22]:
import os
import time
import socket

import numpy as np
import matplotlib.pyplot as plt

import pyvisa
import pyvisa_py

In [41]:
SigGen_IP = '172.31.255.67'
SigGen_PORT = 10001
Scope_IP = '172.31.255.220'

In [42]:
import subprocess

print(subprocess.check_output(
    ['ping', SigGen_IP],
    text=True
))

print(subprocess.check_output(
    ['ping', Scope_IP],
    text=True
))


Pinging 172.31.255.67 with 32 bytes of data:
Reply from 172.31.255.67: bytes=32 time<1ms TTL=64
Reply from 172.31.255.67: bytes=32 time<1ms TTL=64
Reply from 172.31.255.67: bytes=32 time<1ms TTL=64
Reply from 172.31.255.67: bytes=32 time<1ms TTL=64

Ping statistics for 172.31.255.67:
    Packets: Sent = 4, Received = 4, Lost = 0 (0% loss),
Approximate round trip times in milli-seconds:
    Minimum = 0ms, Maximum = 0ms, Average = 0ms


Pinging 172.31.255.220 with 32 bytes of data:
Reply from 172.31.255.220: bytes=32 time<1ms TTL=64
Reply from 172.31.255.220: bytes=32 time<1ms TTL=64
Reply from 172.31.255.220: bytes=32 time<1ms TTL=64
Reply from 172.31.255.220: bytes=32 time<1ms TTL=64

Ping statistics for 172.31.255.220:
    Packets: Sent = 4, Received = 4, Lost = 0 (0% loss),
Approximate round trip times in milli-seconds:
    Minimum = 0ms, Maximum = 0ms, Average = 0ms



In [43]:
rm = pyvisa.ResourceManager('@py')

scope = rm.open_resource(
    f'TCPIP0::{Scope_IP}::INSTR'
)

scope.timeout = 10000

print(scope.query('*IDN?'))

Siglent Technologies,SDS1104X-E,SDSMMGKD804596,8.3.6.1.37R17



In [44]:
siggen = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
siggen.connect((SigGen_IP, SigGen_PORT))
siggen.settimeout(2)

print("Connected")

Connected


In [45]:
def sg_write(cmd):
    siggen.send((cmd + "\r\n").encode())

In [46]:
def sg_query(cmd):
    siggen.send((cmd + "\r\n").encode())
    time.sleep(0.1)
    return siggen.recv(1024).decode(errors="ignore").strip()

In [47]:
print(sg_query("*IDN?"))

DS Instruments,SG22000PRO,4627,V8.50


In [48]:
def sg_set_freq(f):
    sg_write(f"FREQ {int(f)}")

def sg_set_power(p):
    sg_write(f"POW {p}")

def sg_on():
    sg_write("OUTP ON")

def sg_off():
    sg_write("OUTP OFF")

In [49]:
siggen.close()
time.sleep(0.5)

In [55]:
# -----------------------------
# Sweep configuration
# -----------------------------
start_f = 5e9
stop_f  = 7e9
step_f  = 5e8   # 500 MHz

settle_time = 0.25   # adjust if needed

frequencies = np.arange(start_f, stop_f + step_f, step_f)

results = []

# -----------------------------
# Safe SG control helpers
# -----------------------------
def sg_write(cmd):
    siggen.send((cmd + "\r\n").encode())

def sg_set_freq(f):
    sg_write(f"FREQ {int(f)}")

def sg_set_power(p):
    sg_write(f"POW {p}")

def sg_rf_on():
    sg_write("OUTP ON")

def sg_rf_off():
    sg_write("OUTP OFF")

# force clean reconnect
try:
    siggen.close()
except:
    pass

siggen = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
siggen.connect((SigGen_IP, SigGen_PORT))
siggen.settimeout(2)

print("SigGen reconnected")


# -----------------------------
# Sweep start
# -----------------------------
sg_rf_on()
sg_set_power(10)

print("Starting sweep...\n")

for f in frequencies:

    sg_set_freq(f)
    time.sleep(settle_time)

    try:
        resp = scope.query("C1:PAVA? PKPK").strip()

        import re
        match = re.search(r"[-+]?\d*\.?\d+([eE][-+]?\d+)?", resp)

        vpp = float(match.group()) if match else float('nan')

    except Exception as e:
        vpp = np.nan
        print(f"Scope error at {f/1e9:.2f} GHz:", e)

    results.append((f, vpp))

    print(f"{f/1e9:.2f} GHz → {vpp}")

sg_rf_off()

print("\nSweep complete.")

SigGen reconnected
Starting sweep...

5.00 GHz → 1.0
5.50 GHz → 1.0
6.00 GHz → 1.0
6.50 GHz → 1.0
7.00 GHz → 1.0

Sweep complete.
